# Tutorial 2: Conditional Molecule Generation

This tutorial demonstrates conditional generation - generating molecules with specific properties.

**Time to complete**: 20-30 minutes

**What you'll learn**:
- Train a property-conditioned model
- Generate molecules with target properties
- Evaluate property accuracy
- Use exact conditional generation

**Prerequisites**: Complete Tutorial 1

## Setup

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import argparse

from qm9 import dataset
from qm9.models import get_model, get_optim
from qm9.utils import prepare_context, compute_mean_mad
from configs.datasets_config import get_dataset_info
from equivariant_diffusion.utils import remove_mean_with_mask

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Property-Conditioned Training

We'll condition on molecular polarizability (alpha).

In [ ]:
# Setup arguments
args = argparse.Namespace()

# Dataset
args.dataset = 'qm9'
args.batch_size = 32
args.num_workers = 2
args.filter_n_atoms = 10  # Small molecules for quick training
args.datadir = 'data'
args.remove_h = False
args.include_charges = True

# Conditioning - THIS IS KEY!
args.conditioning = ['alpha']  # Polarizability

# Model
args.model = 'egnn_dynamics'
args.probabilistic_model = 'diffusion'
args.nf = 128
args.n_layers = 6
args.attention = True
args.tanh = True
args.norm_constant = 1
args.inv_sublayers = 1
args.sin_embedding = False
args.normalization_factor = 1
args.aggregation_method = 'sum'

# Diffusion
args.diffusion_steps = 500
args.diffusion_noise_schedule = 'polynomial_2'
args.diffusion_noise_precision = 1e-5
args.diffusion_loss_type = 'l2'
args.normalize_factors = [1, 4, 1]

# Training
args.n_epochs = 5  # Quick demo
args.lr = 2e-4
args.ema_decay = 0.999
args.clip_grad = True
args.clip_grad_norm = 1.0

args.device = device

print("Loading dataset with properties...")
dataloaders, charge_scale = dataset.retrieve_dataloaders(args)
dataset_info = get_dataset_info('qm9', remove_h=args.remove_h)

print(f"Training set size: {len(dataloaders['train'].dataset)}")
print(f"Conditioning on: {args.conditioning}")

### Compute Property Normalization

We normalize properties to zero mean and unit variance for stable training.

In [ ]:
# Compute mean and MAD (mean absolute deviation)
property_norms = compute_mean_mad(dataloaders, args.conditioning, args.dataset)

print("Property normalization:")
for prop in args.conditioning:
    print(f"  {prop}:")
    print(f"    Mean: {property_norms[prop]['mean']:.4f}")
    print(f"    MAD: {property_norms[prop]['mad']:.4f}")

In [ ]:
# Visualize property distribution
alphas = []
for data in dataloaders['train']:
    if 'alpha' in data:
        alphas.extend(data['alpha'].numpy())

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(alphas, bins=50, alpha=0.7, edgecolor='black')
plt.xlabel('Alpha (Bohr³)')
plt.ylabel('Count')
plt.title('Distribution of Polarizability')
plt.grid(True, alpha=0.3)

# Normalized
mean = property_norms['alpha']['mean']
mad = property_norms['alpha']['mad']
alphas_norm = [(a - mean) / mad for a in alphas]

plt.subplot(1, 2, 2)
plt.hist(alphas_norm, bins=50, alpha=0.7, edgecolor='black', color='orange')
plt.xlabel('Normalized Alpha')
plt.ylabel('Count')
plt.title('Normalized Distribution')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Alpha range: [{min(alphas):.2f}, {max(alphas):.2f}]")
print(f"Normalized range: [{min(alphas_norm):.2f}, {max(alphas_norm):.2f}]")

### Create Conditional Model

In [ ]:
# Context dimension = number of conditioning properties
args.context_node_nf = len(args.conditioning)

print(f"Creating conditional model with context_dim={args.context_node_nf}...")
model, nodes_dist, prop_dist = get_model(args, device, dataset_info, dataloaders['train'])
model = model.to(device)

# Set property normalization
if prop_dist is not None:
    prop_dist.set_normalizer(property_norms)

optimizer = get_optim(args, model)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")

### Train Conditional Model

In [ ]:
print("Training conditional model...")

train_losses = []
val_losses = []

for epoch in range(args.n_epochs):
    model.train()
    epoch_loss = 0
    n_batches = 0
    
    for batch_idx, data in enumerate(dataloaders['train']):
        # Move to device
        x = data['positions'].to(device, torch.float32)
        node_mask = data['atom_mask'].to(device, torch.float32)
        edge_mask = data['edge_mask'].to(device, torch.float32)
        one_hot = data['one_hot'].to(device, torch.float32)
        charges = data['charges'].to(device, torch.float32) if args.include_charges else torch.zeros(0)
        
        # Prepare context (property values)
        context = prepare_context(args.conditioning, data, property_norms).to(device, torch.float32)
        
        # Remove center of mass
        x = remove_mean_with_mask(x, node_mask)
        h = {'categorical': one_hot, 'integer': charges}
        
        # Forward pass
        optimizer.zero_grad()
        nll, reg_term, mean_abs_z = model(x, h, node_mask, edge_mask, context=context)
        loss = nll.mean()
        
        # Backward
        loss.backward()
        if args.clip_grad:
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip_grad_norm)
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
        
        if batch_idx >= 10:  # Quick demo
            break
    
    avg_train_loss = epoch_loss / n_batches
    train_losses.append(avg_train_loss)
    
    # Validation
    model.eval()
    val_loss = 0
    n_val_batches = 0
    
    with torch.no_grad():
        for batch_idx, data in enumerate(dataloaders['valid']):
            x = data['positions'].to(device, torch.float32)
            node_mask = data['atom_mask'].to(device, torch.float32)
            edge_mask = data['edge_mask'].to(device, torch.float32)
            one_hot = data['one_hot'].to(device, torch.float32)
            charges = data['charges'].to(device, torch.float32) if args.include_charges else torch.zeros(0)
            context = prepare_context(args.conditioning, data, property_norms).to(device, torch.float32)
            
            x = remove_mean_with_mask(x, node_mask)
            h = {'categorical': one_hot, 'integer': charges}
            
            nll, reg_term, mean_abs_z = model(x, h, node_mask, edge_mask, context=context)
            val_loss += nll.mean().item()
            n_val_batches += 1
            
            if batch_idx >= 5:
                break
    
    avg_val_loss = val_loss / n_val_batches
    val_losses.append(avg_val_loss)
    
    print(f"Epoch {epoch+1}/{args.n_epochs} - Train: {avg_train_loss:.4f}, Val: {avg_val_loss:.4f}")

print("Training complete!")

## 2. Conditional Sampling

Generate molecules with specific property values.

In [ ]:
print("Generating molecules with different alpha values...")

model.eval()
n_samples = 3
n_nodes = 8

# Target alpha values (unnormalized)
target_alphas = [50.0, 75.0, 100.0]  # Low, medium, high

# Normalize
mean = property_norms['alpha']['mean']
mad = property_norms['alpha']['mad']
target_alphas_norm = [(a - mean) / mad for a in target_alphas]

print("\nTarget values:")
for i, (alpha, alpha_norm) in enumerate(zip(target_alphas, target_alphas_norm)):
    print(f"  Sample {i+1}: alpha = {alpha:.1f} (normalized: {alpha_norm:.2f})")

# Generate
generated_molecules = []

with torch.no_grad():
    for i, alpha_norm in enumerate(target_alphas_norm):
        # Create context with target property value
        context = torch.tensor([[alpha_norm]], dtype=torch.float32, device=device)
        context = context.expand(1, n_nodes, -1)  # [1, n_nodes, 1]
        
        # Create masks
        node_mask = torch.ones(1, n_nodes, 1, device=device)
        edge_mask = torch.ones(1, n_nodes, n_nodes, device=device)
        
        # Sample
        x, h = model.sample(1, n_nodes, node_mask, edge_mask, context=context)
        
        generated_molecules.append({
            'positions': x[0].cpu().numpy(),
            'atom_types': h['categorical'][0].argmax(dim=-1).cpu().numpy(),
            'target_alpha': target_alphas[i],
        })

print(f"\nGenerated {len(generated_molecules)} molecules")

In [ ]:
# Visualize generated molecules
fig = plt.figure(figsize=(15, 4))

colors = ['red', 'gray', 'blue', 'darkred', 'yellow']

for i, mol in enumerate(generated_molecules):
    ax = fig.add_subplot(1, 3, i+1, projection='3d')
    
    positions = mol['positions']
    atom_types = mol['atom_types']
    
    for atom_type in range(len(dataset_info['atom_decoder'])):
        mask = atom_types == atom_type
        if mask.any():
            ax.scatter(positions[mask, 0], positions[mask, 1], positions[mask, 2],
                      c=colors[atom_type], s=150, alpha=0.8)
    
    ax.set_title(f'Target α = {mol["target_alpha"]:.1f}')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    
    # Composition
    composition = [dataset_info['atom_decoder'][t] for t in atom_types]
    print(f"\nMolecule {i+1} (α={mol['target_alpha']:.1f}):")
    print(f"  Composition: {composition}")

plt.tight_layout()
plt.show()

## 3. Property Sweep

Generate molecules across a range of property values.

In [ ]:
print("Generating property sweep...")

# Define range
alpha_min = 40.0
alpha_max = 120.0
n_bins = 5

alpha_values = np.linspace(alpha_min, alpha_max, n_bins)
n_samples_per_bin = 2

sweep_results = []

with torch.no_grad():
    for alpha in alpha_values:
        # Normalize
        alpha_norm = (alpha - mean) / mad
        
        # Generate multiple samples
        for _ in range(n_samples_per_bin):
            context = torch.tensor([[alpha_norm]], dtype=torch.float32, device=device)
            context = context.expand(1, n_nodes, -1)
            
            node_mask = torch.ones(1, n_nodes, 1, device=device)
            edge_mask = torch.ones(1, n_nodes, n_nodes, device=device)
            
            x, h = model.sample(1, n_nodes, node_mask, edge_mask, context=context)
            
            sweep_results.append({
                'alpha': alpha,
                'alpha_norm': alpha_norm,
                'positions': x[0].cpu().numpy(),
                'atom_types': h['categorical'][0].argmax(dim=-1).cpu().numpy(),
            })

print(f"Generated {len(sweep_results)} molecules across {n_bins} property bins")

In [ ]:
# Visualize sweep
fig, axes = plt.subplots(n_bins, n_samples_per_bin, figsize=(8, 10), subplot_kw={'projection': '3d'})
if n_samples_per_bin == 1:
    axes = axes.reshape(-1, 1)

idx = 0
for i in range(n_bins):
    for j in range(n_samples_per_bin):
        ax = axes[i, j]
        mol = sweep_results[idx]
        
        positions = mol['positions']
        atom_types = mol['atom_types']
        
        for atom_type in range(len(dataset_info['atom_decoder'])):
            mask = atom_types == atom_type
            if mask.any():
                ax.scatter(positions[mask, 0], positions[mask, 1], positions[mask, 2],
                          c=colors[atom_type], s=50, alpha=0.8)
        
        if j == 0:
            ax.set_ylabel(f'α={mol["alpha"]:.1f}', fontsize=10)
        
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_zticks([])
        
        idx += 1

plt.suptitle('Property Sweep: Polarizability', fontsize=14, y=0.995)
plt.tight_layout()
plt.show()

## 4. Multiple Property Conditioning

Example: Condition on both alpha and gap (HOMO-LUMO gap).

In [ ]:
print("Example: Multiple property conditioning")
print("\nTo condition on multiple properties:")
print("""\nargs.conditioning = ['alpha', 'gap']
args.context_node_nf = len(args.conditioning)  # 2

# Training data includes both properties
context = prepare_context(args.conditioning, data, property_norms)
# context shape: [batch, n_nodes, 2]

# Sampling with target values
target_alpha = 75.0
target_gap = 0.25

alpha_norm = (target_alpha - property_norms['alpha']['mean']) / property_norms['alpha']['mad']
gap_norm = (target_gap - property_norms['gap']['mean']) / property_norms['gap']['mad']

context = torch.tensor([[[alpha_norm, gap_norm]]], device=device)
context = context.expand(1, n_nodes, -1)

x, h = model.sample(1, n_nodes, node_mask, edge_mask, context=context)
""")

print("\nThis allows precise control over multiple molecular properties!")

## 5. Exact Conditional Generation (ASE Databases)

For molecular descriptors with ASE databases.

In [ ]:
print("Exact conditional generation with molecular descriptors:")
print("\nAvailable descriptors for ASE databases:")
print("  - molecular_weight: Molecular mass (u)")
print("  - pi_conjugation_ratio: π bond ratio [0, 1]")
print("  - atom_types_encoding: Binary encoding of elements")
print("  - functional_groups_encoding: SMARTS pattern matching")

print("\nExample usage:")
print("""\n# Train with molecular descriptors
python main_qm9.py \\
    --dataset ase_db \\
    --ase_db_path my_database.db \\
    --conditioning molecular_weight pi_conjugation_ratio atom_types_encoding \\
    --exp_name descriptor_model

# Generate with exact values
python eval_conditional_qm9.py \\
    --generators_path outputs/descriptor_model \\
    --use_exact_conditions \\
    --property_values "molecular_weight=50.0,pi_conjugation_ratio=0.9,atom_types_encoding=[C,H,N,O]"
""")

## Summary

In this tutorial, you learned:

1. ✅ How to train property-conditioned models
2. ✅ How to normalize properties for training
3. ✅ How to generate molecules with target properties
4. ✅ How to create property sweeps
5. ✅ How to condition on multiple properties
6. ✅ Exact conditional generation with descriptors

## Key Concepts

**Property Normalization**: Essential for stable training
- Compute mean and MAD (mean absolute deviation)
- Normalize: `(value - mean) / mad`
- Denormalize for interpretation

**Context Vector**: Property values passed to model
- Shape: `[batch, n_nodes, n_properties]`
- Expanded to all nodes (homogeneous conditioning)
- Can condition on 1 or multiple properties

**Sampling**: Control generation precisely
- Set target property values
- Model generates molecules matching targets
- Quality depends on training

## Next Steps

- **Tutorial 3**: Crystal generation
- **Tutorial 4**: Molecular descriptors with ASE
- **Tutorial 5**: Evaluation and analysis

## For Production

```bash
# Train high-quality conditional model
python main_qm9.py \
    --exp_name conditional_alpha \
    --conditioning alpha \
    --n_epochs 1000 \
    --batch_size 64 \
    --lr 1e-4 \
    --nf 192 \
    --n_layers 9 \
    --diffusion_steps 1000
```